# Caching functions that read and write files

`fleche` is a content-addressed cache.  When a cached function takes or returns a
`pathlib.Path`, fleche stores the file's (or directory tree's) **contents** — not
just the path string — so results are portable and reproducible across machines.

A path is keyed by its content, so:

- two files with identical bytes share one storage entry (deduplication), and
- re-calling a function with the same file content is a cache hit, even if the
  file lives at a different location.

The in-memory cache (`cache("memory")`) and every default value storage now carry
this behaviour out of the box — no custom storage composition required.

In [ ]:
import tempfile
from pathlib import Path
from subprocess import run, CompletedProcess

import fleche as fl
from fleche import fleche

fl.cache("memory")          # activate a transient in-memory cache
c = fl.cache()              # grab the active cache to introspect later

# The default value storage stores Paths by content:
[k.__name__ for k in type(c.values).__mro__ if k.__name__.endswith("Mixin")]

In [ ]:
# A scratch directory for the files our functions produce.
WORK = Path(tempfile.mkdtemp(suffix="-fleche-files"))
WORK

## Producing a file

A cached function can create a file and return its `Path`.  fleche stores the
bytes; on a cache hit it hands back a freshly materialized temporary `Path` with
the same contents.  The `print` fires only when the body actually runs.

In [ ]:
@fleche
def write(text, repeat=1, name="out.txt"):
    print("  [write] running:", repr(text))
    f = WORK / name
    f.write_text(text * repeat)
    return f

f = write("hello", 2)
print("returned:", type(f).__name__, "->", f.read_text())

In [ ]:
# Same arguments -> cache hit -> the body does NOT run (no "[write] running").
again = write("hello", 2)
print("from cache:", again.read_text())

## Consuming a file

A function can take a `Path` argument; fleche keys the call on the file's
content.  Feeding it a path produced by another cached function chains the two.

In [ ]:
@fleche
def wordcount(path: Path):
    print("  [wordcount] running:", path.name)
    return len(path.read_text().split())

print("count:", wordcount(write("a quick brown fox", 1, name="sentence.txt")))
# Re-run: both `write` and `wordcount` are served from cache.
print("count again:", wordcount(write("a quick brown fox", 1, name="sentence.txt")))

## Directories

Returning a directory `Path` stores the whole tree.  On load it is rebuilt under
a temporary directory, so `iterdir()` / `rglob()` work exactly as before.

In [ ]:
@fleche
def make_tree(seed):
    print("  [make_tree] running:", seed)
    d = WORK / f"tree-{seed}"
    d.mkdir(exist_ok=True)
    (d / "top.txt").write_text(seed)
    (d / "sub").mkdir(exist_ok=True)
    (d / "sub" / "leaf.bin").write_bytes(seed.encode() * 3)
    return d

@fleche
def total_bytes(d: Path):
    print("  [total_bytes] running:", d.name)
    return sum(p.stat().st_size for p in d.rglob("*") if p.is_file())

tree = make_tree("alpha")
sorted(p.relative_to(tree).as_posix() for p in tree.rglob("*"))

In [ ]:
# End-to-end cache hit: make_tree("alpha") and total_bytes both come from cache.
total_bytes(make_tree("alpha"))

## Content-addressing

Files are stored under the digest of their contents, so identical bodies are
stored once regardless of filename or which call produced them.

In [ ]:
# Two calls, different names, identical body -> the content is stored once.
write("shared body", 1, name="left.txt")
write("shared body", 1, name="right.txt")

body_key = fl.digest.digest(b"shared body")     # content blobs are plain bytes
print("shared body stored once:", body_key in set(c.values.list()))

In [ ]:
# Introspect the recorded calls of any cached function.
write.query().table()

## A real workflow: orchestrating shell scripts

The motivating use case: wrap command-line tools that read and write files.  Each
step runs in a working directory, produces files, and the whole pipeline is
cached by content.

A `subprocess.CompletedProcess` isn't digestible out of the box, so we register a
digest hook describing how to fingerprint one.

In [ ]:
def digest_completedprocess(cp):
    return fl.digest.digest((type(cp).__name__, cp.args, cp.returncode, cp.stdout, cp.stderr))

fl.digest.add_hook((CompletedProcess, digest_completedprocess))

@fleche
def shell(cwd, prog, args=(), stdin=b""):
    print("  [shell] running:", prog, *args)
    ret = run([prog, *args], cwd=cwd, capture_output=True, input=stdin)
    return cwd, ret

@fleche
def pipeline(content):
    print("  [pipeline] running:", content)
    work = Path(tempfile.mkdtemp(suffix="-pipeline"))
    (work / "input.txt").write_bytes(content)
    shell(work, "cp", ["input.txt", "copy.txt"])                       # produce a file
    _, upper = shell(work, "tr", ["a-z", "A-Z"], stdin=content)        # capture stdout
    (work / "shout.txt").write_bytes(upper.stdout)
    return work

In [ ]:
print("--- first run ---")
out = pipeline(b"hello world")
print("produced:", sorted(p.name for p in out.iterdir()))
print("shout.txt:", (out / "shout.txt").read_text())

In [ ]:
print("--- second run: fully cached (no body / shell prints) ---")
out2 = pipeline(b"hello world")
print("same files:", sorted(p.name for p in out2.iterdir()))

In [ ]:
# Every shell invocation fleche recorded:
shell.query().table()